# Defining Prompts

**Prompts** let an MCP server ship pre-built, high-quality instructions that clients use instead of writing their own from scratch — carefully crafted templates that give better, more consistent results.

> Implement this in **`cli-project/mcp_server.py`** (the prompt `TODO`s). `skip-worktree` keeps your edits local; answer key in `cli-project-complete/`.

## Why prompts?

Users can already ask Claude directly — *"reformat report.pdf in markdown"* gives decent results. But a **tested, specialized prompt** that handles edge cases and best practices does much better.

As the **server author** you craft, test, and evaluate prompts once; users get that expertise without becoming prompt engineers. The value lives with whoever knows the domain.

## Example: a `/format` command

Convert documents to markdown. Users type **`/format doc_id`** and get a professionally formatted version.

1. User types `/` to see available commands.
2. Selects **format** and gives a document id.
3. Claude uses your pre-built prompt to read and reformat the document.
4. Result is clean markdown — headers, lists, tables.

## Defining a prompt

Same decorator pattern as tools and resources:

```python
@mcp.prompt(
    name="format",
    description="Rewrites the contents of the document in Markdown format."
)
def format_document(
    doc_id: str = Field(description="Id of the document to format")
) -> list[base.Message]:
    prompt = f"""
Your goal is to reformat a document to be written with markdown syntax.

The id of the document you need to reformat is:
<document_id>
{doc_id}
</document_id>

Add in headers, bullet points, tables, etc as necessary. Feel free to add in structure.
Use the 'edit_document' tool to edit the document. After the document has been reformatted...
"""

    return [
        base.UserMessage(prompt)
    ]
```

The function returns a **list of messages** sent directly to Claude. Include multiple user/assistant messages for more complex conversation flows. (Note the prompt tells Claude to use the `edit_document` **tool** — prompts, tools, and resources compose.)

> Import note: `base` comes from `mcp.server.fastmcp.prompts` (e.g. `from mcp.server.fastmcp.prompts import base`), and `Field` from `pydantic`.

## Testing prompts

Use the MCP Inspector. It shows **exactly what messages** will be sent to Claude, including how variables interpolate into the template — so you verify the prompt before users rely on it.

## Key benefits

- **Consistency** — reliable results every time.
- **Expertise** — encode domain knowledge into the prompt.
- **Reusability** — multiple client apps share the same prompts.
- **Maintenance** — update in one place, every client improves.

Prompts work best when **specialized to your server's domain**: a document server has format/summarize/analyze prompts; a data-analysis server has report/visualization prompts. The goal: prompts so well-crafted that users prefer them over writing their own.

Next: **Prompts in the client** — discover and invoke these prompts from the client side (`list_prompts` / `get_prompt`).